## Ex:2 Data Wrangling and Transformation 
### Objective

To perform data wrangling and transformation on a dataset using Python and Pandas by handling missing values, removing duplicates, correcting data types, filtering data, and transforming variables into a suitable format for data analysis and machine learning.



##  Dataset Description

The dataset contains information about **student academic performance, educational background, MBA specialization, and placement salary**.

###  Dataset Attributes

| Column | Description |
|:---|:---|
| `sl_no` | Serial number / student identifier |
| `gender` | Gender of the student |
| `hsc_p` | Higher Secondary / 12th percentage |
| `hsc_s` | Higher Secondary stream |
| `degree_p` | Undergraduate degree percentage |
| `degree_t` | Undergraduate degree type |
| `etest_p` | Employability / entrance test percentage |
| `specialisation` | MBA specialization |
| `mba_p` | MBA percentage |
| `salary` | Salary offered after placement |

---

## Experiment Question

 **Using the given student placement dataset, perform data wrangling and transformation by handling missing values, scaling numerical features, detecting and treating outliers, encoding categorical variables, and generating a final model-ready dataset.**




Data wrangling and transformation is the process of converting raw student placement data into a clean, consistent, and machine-learning-ready dataset. In this experiment, the given student placement dataset is first loaded into a Pandas DataFrame and explored by examining its rows, columns, data types, and descriptive statistics. Missing values are then identified and handled by removing records with missing `salary` values and replacing missing values in `hsc_p`, `degree_p`, and `etest_p` with their respective mean values. The numerical attributes such as `hsc_p`, `degree_p`, `etest_p`, and `salary` are transformed using feature-scaling techniques such as `StandardScaler` and `MinMaxScaler` to bring the variables into suitable numerical ranges. The dataset is then divided into input features (`X`) and the target variable (`Y`), where `salary` is considered the target. The preprocessed data is saved as `Pre.csv` for further processing. Next, possible outliers in the `salary` attribute are identified using a boxplot and statistically detected using the Z-score method, where values with an absolute Z-score greater than 3 are considered potential outliers. Outliers are also treated using the capping and flooring method by calculating the 5th and 95th percentiles and replacing values outside these limits with the corresponding boundary values. The salary distribution before and after outlier treatment is then compared using visualization. Since the dataset also contains categorical attributes such as `gender`, `hsc_s`, `degree_t`, and `specialisation`, these variables are converted into numerical representations using `LabelEncoder` and One-Hot Encoding. Finally, the completely transformed dataset is verified for missing values, data types, dimensions, and numerical representation, and the resulting model-ready dataset is saved as `Final.csv`. Thus, the experiment demonstrates the complete workflow of preparing real-world tabular data for data analysis and machine-learning applications.

## Step 1: Start by importing the necessary Python libraries for data preprocessing.


In [1]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.preprocessing import LabelEncoder
from scipy.stats import zscore
from scipy import stats


## Step 2: Load the placement dataset into a Pandas Dataframe.

In [3]:

df=pd.read_csv("data.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 215 entries, 0 to 214
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sl_no           215 non-null    int64  
 1   gender          215 non-null    str    
 2   hsc_p           210 non-null    float64
 3   hsc_s           215 non-null    str    
 4   degree_p        213 non-null    float64
 5   degree_t        215 non-null    str    
 6   etest_p         211 non-null    float64
 7   specialisation  215 non-null    str    
 8   mba_p           214 non-null    float64
 9   salary          148 non-null    float64
dtypes: float64(5), int64(1), str(4)
memory usage: 16.9 KB


In [4]:
df.shape

(215, 10)

In [5]:
df.head

<bound method NDFrame.head of      sl_no gender  hsc_p     hsc_s  degree_p   degree_t  etest_p  \
0        1      M  91.00  Commerce     58.00   Sci&Tech     55.0   
1        2      M  78.33   Science     77.48   Sci&Tech     86.5   
2        3      M    NaN      Arts     64.00  Comm&Mgmt     75.0   
3        4      M  52.00   Science       NaN   Sci&Tech     66.0   
4        5      M  73.60  Commerce     73.30  Comm&Mgmt     96.8   
..     ...    ...    ...       ...       ...        ...      ...   
210    211      M  82.00  Commerce     77.60  Comm&Mgmt     91.0   
211    212      M  60.00   Science     72.00   Sci&Tech     74.0   
212    213      M  67.00  Commerce     73.00  Comm&Mgmt     59.0   
213    214      F  66.00  Commerce     58.00  Comm&Mgmt     70.0   
214    215      M  58.00   Science     53.00  Comm&Mgmt     89.0   

    specialisation  mba_p    salary  
0           Mkt&HR  58.80  270000.0  
1          Mkt&Fin  66.28  200000.0  
2          Mkt&Fin  57.80  250000.0  
3

In [6]:
df.tail

<bound method NDFrame.tail of      sl_no gender  hsc_p     hsc_s  degree_p   degree_t  etest_p  \
0        1      M  91.00  Commerce     58.00   Sci&Tech     55.0   
1        2      M  78.33   Science     77.48   Sci&Tech     86.5   
2        3      M    NaN      Arts     64.00  Comm&Mgmt     75.0   
3        4      M  52.00   Science       NaN   Sci&Tech     66.0   
4        5      M  73.60  Commerce     73.30  Comm&Mgmt     96.8   
..     ...    ...    ...       ...       ...        ...      ...   
210    211      M  82.00  Commerce     77.60  Comm&Mgmt     91.0   
211    212      M  60.00   Science     72.00   Sci&Tech     74.0   
212    213      M  67.00  Commerce     73.00  Comm&Mgmt     59.0   
213    214      F  66.00  Commerce     58.00  Comm&Mgmt     70.0   
214    215      M  58.00   Science     53.00  Comm&Mgmt     89.0   

    specialisation  mba_p    salary  
0           Mkt&HR  58.80  270000.0  
1          Mkt&Fin  66.28  200000.0  
2          Mkt&Fin  57.80  250000.0  
3

In [7]:
df.sample(5)

,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
84,85,M,63.00,Science,70.0,Sci&Tech,55.00,Mkt&Fin,62.00,300000.0
51,52,M,61.12,Commerce,56.2,Comm&Mgmt,67.00,Mkt&HR,62.65,NaN
113,114,F,79.00,Commerce,67.0,Comm&Mgmt,72.15,Mkt&Fin,63.08,280000.0
202,203,M,63.00,Science,66.0,Sci&Tech,61.28,Mkt&HR,60.11,240000.0
74,75,M,64.80,Commerce,70.2,Comm&Mgmt,84.27,Mkt&Fin,67.20,336000.0


In [8]:
df.describe()

,sl_no,hsc_p,degree_p,etest_p,mba_p,salary
count,215.000000,210.000000,213.000000,211.000000,214.000000,148.000000
mean,108.000000,66.503000,66.420610,72.093649,62.254813,288655.405405
std,62.209324,10.904205,7.322786,13.340195,5.836962,93457.452420
min,1.000000,37.000000,50.000000,50.000000,51.210000,200000.000000
25%,54.500000,61.000000,61.000000,60.000000,57.922500,240000.000000
50%,108.000000,65.000000,66.000000,70.000000,61.950000,265000.000000
75%,161.500000,73.000000,72.000000,84.000000,66.187500,300000.000000
max,215.000000,97.700000,91.000000,98.000000,77.890000,940000.000000


In [9]:
df.loc[0]

sl_no                    1
gender                   M
hsc_p                 91.0
hsc_s             Commerce
degree_p              58.0
degree_t          Sci&Tech
etest_p               55.0
specialisation      Mkt&HR
mba_p                 58.8
salary            270000.0
Name: 0, dtype: object

In [10]:
df.iloc[0]

sl_no                    1
gender                   M
hsc_p                 91.0
hsc_s             Commerce
degree_p              58.0
degree_t          Sci&Tech
etest_p               55.0
specialisation      Mkt&HR
mba_p                 58.8
salary            270000.0
Name: 0, dtype: object

In [11]:
df[0:2]

,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
0,1,M,91.00,Commerce,58.00,Sci&Tech,55.0,Mkt&HR,58.80,270000.0
1,2,M,78.33,Science,77.48,Sci&Tech,86.5,Mkt&Fin,66.28,200000.0


In [12]:
df["degree_p"]

0      58.00
1      77.48
2      64.00
3        NaN
4      73.30
       ...  
210    77.60
211    72.00
212    73.00
213    58.00
214    53.00
Name: degree_p, Length: 215, dtype: float64

In [13]:
df.columns


Index(['sl_no', 'gender', 'hsc_p', 'hsc_s', 'degree_p', 'degree_t', 'etest_p',
       'specialisation', 'mba_p', 'salary'],
      dtype='str')

## Step 3:Take a quick look at the data to understand its structure and identify any missing values or anomalies.

In [14]:
df.dropna(subset=["salary"],inplace=True)

In [15]:
df.isnull().sum()

sl_no             0
gender            0
hsc_p             2
hsc_s             0
degree_p          1
degree_t          0
etest_p           2
specialisation    0
mba_p             0
salary            0
dtype: int64

In [16]:
df.dtypes


sl_no               int64
gender                str
hsc_p             float64
hsc_s                 str
degree_p          float64
degree_t              str
etest_p           float64
specialisation        str
mba_p             float64
salary            float64
dtype: object

In [17]:
df.duplicated().sum()


np.int64(0)

In [18]:
df.nunique()


sl_no             148
gender              2
hsc_p              71
hsc_s               3
degree_p           69
degree_t            3
etest_p            78
specialisation      2
mba_p             143
salary             45
dtype: int64

In [19]:
df["gender"].unique()


<StringArray>
['M', 'F']
Length: 2, dtype: str

In [20]:
print("hsc_s unique:", df["hsc_s"].unique())
print("degree_t unique:", df["degree_t"].unique())
print("specialisation unique:", df["specialisation"].unique())


hsc_s unique: <StringArray>
['Commerce', 'Science', 'Arts']
Length: 3, dtype: str
degree_t unique: <StringArray>
['Sci&Tech', 'Comm&Mgmt', 'Others']
Length: 3, dtype: str
specialisation unique: <StringArray>
['Mkt&HR', 'Mkt&Fin']
Length: 2, dtype: str


In [21]:
df.isnull().sum()


sl_no             0
gender            0
hsc_p             2
hsc_s             0
degree_p          1
degree_t          0
etest_p           2
specialisation    0
mba_p             0
salary            0
dtype: int64

#### The method isnull() checks each element in the DataFrame (or Series) to see if it is NaN (Not a Number) or None (missing value).
It returns a DataFrame (or Series) of the same shape as the input, with Boolean values:
#### True: The value is null (NaN or None).
#### False: The value is not null.

In [22]:
df.isnull().sum()


sl_no             0
gender            0
hsc_p             2
hsc_s             0
degree_p          1
degree_t          0
etest_p           2
specialisation    0
mba_p             0
salary            0
dtype: int64

## Step 4: Handle Missing Data
### Option 1: If the dataset is large and only a small percentage of data is missing, you can remove rows with missing values using dropna(subset,inplace)


In [4]:
df["hsc_p"]=df["hsc_p"].fillna(df["hsc_p"].mean())

NameError: name 'df' is not defined

In [24]:
df["degree_p"]=df["degree_p"].fillna(df["degree_p"].mean())


In [25]:
df["etest_p"]=df["etest_p"].fillna(df["etest_p"].mean())

### Option 2:If removing data isn't ideal, you can impute (df.[""].fillna(df[""].mean(),inplace)) missing values using methods like mean, median, or most frequent.

In [3]:
df["mba_p"].fillna(df["mba_p"].median(), inplace=True)
df.isnull().sum()


NameError: name 'df' is not defined

## Step 5: Feature Scaling
Feature scaling is the process of converting numerical features to a similar scale so that one feature does not dominate another simply because it has larger numerical values.

<img src="https://i.postimg.cc/G21gMYnF/f.png" alt="Image Description" width="500">









## Option 1( StandardScaler): This method scales the data to have a mean of 0 and a standard deviation of 1.


In [27]:
c=["hsc_p","degree_p","etest_p","salary"]
s1=StandardScaler()
df[c]=s1.fit_transform(df[c])
df.head()



,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
0,1,M,2.265997e+00,Commerce,-1.652293,Sci&Tech,-1.328518,Mkt&HR,58.80,-0.200292
1,2,M,8.987875e-01,Science,1.346845,Sci&Tech,0.978332,Mkt&Fin,66.28,-0.951839
2,3,M,1.533482e-15,Arts,-0.728534,Comm&Mgmt,0.136149,Mkt&Fin,57.80,-0.415019
4,5,M,3.883770e-01,Commerce,0.703293,Comm&Mgmt,1.732635,Mkt&Fin,55.50,1.463849
7,8,M,-6.475513e-01,Science,-0.420614,Sci&Tech,-0.449718,Mkt&Fin,62.14,-0.393547


#### Option 2:This method scales the data to a fixed range, usually between 0 and 1. 
###  MinMaxScaler()

In [28]:
s2=MinMaxScaler()
df[c]=s2.fit_transform(df[c])
df.head()



,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
0,1,M,0.857051,Commerce,0.057143,Sci&Tech,0.104167,Mkt&HR,58.80,0.094595
1,2,M,0.586729,Science,0.613714,Sci&Tech,0.760417,Mkt&Fin,66.28,0.000000
2,3,M,0.409023,Arts,0.228571,Comm&Mgmt,0.520833,Mkt&Fin,57.80,0.067568
4,5,M,0.485812,Commerce,0.494286,Comm&Mgmt,0.975000,Mkt&Fin,55.50,0.304054
7,8,M,0.280990,Science,0.285714,Sci&Tech,0.354167,Mkt&Fin,62.14,0.070270


## Step 6  Option 1: Identifying Outliers Using Z-Scores
The value of 3 in the context of Z-scores is often used as a threshold to identify outliers in a dataset. A Z-score represents how many standard deviations a data point is away from the mean of the dataset. Specifically:

A Z-score of 0 means the data point is exactly at the mean.
A Z-score of 1 means the data point is one standard deviation above the mean, and so on.
A Z-score of 3 corresponds to a data point being 3 standard deviations away from the mean. For a normal distribution, about 99.7% of the data points fall within 3 standard deviations of the mean (according to the 68-95-99.7 rule, which describes the spread of data in a normal distribution). Therefore, points with Z-scores greater than 3 or less than -3 are considered unusually far from the mean and are often flagged as outliers.

This threshold (Z > 3 or Z < -3) is commonly used in many statistical applications because it captures the extreme values that are rare in a normal distribution, which are typically considered to be outliers. However, the choice of threshold can vary depending on the specific application and the nature of the data.



[![Chat-GPT-Image-Aug-20-2026-08-06-05-PM.png](https://i.postimg.cc/2SFTtcXL/Chat-GPT-Image-Aug-20-2026-08-06-05-PM.png)](https://postimg.cc/4Yyz75GX)

In [29]:
columns_to_check=["salary"]
zscore=stats.zscore(df[columns_to_check])
u=(zscore>3)
l=(zscore<-3)
print(u)
print(l)
indx=u | l
clean_df=df[~indx]
df.info
clean_df.info()

[[False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [ True]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [False]
 [ True]
 [False]
 [False]
 

### Option 2:  Capping and Flooring Outliers
Capping and flooring is an outlier-treatment technique where extreme values are replaced with predefined boundary values instead of deleting the records.

In [30]:
l1=df["salary"].quantile(0.05)
u1=df["salary"].quantile(0.95)
df_capped=df.copy()
df_capped["salary"].clip(l1,u1)
df_capped.info()
df.head()


<class 'pandas.DataFrame'>
Index: 148 entries, 0 to 213
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sl_no           148 non-null    int64  
 1   gender          148 non-null    str    
 2   hsc_p           148 non-null    float64
 3   hsc_s           148 non-null    str    
 4   degree_p        148 non-null    float64
 5   degree_t        148 non-null    str    
 6   etest_p         148 non-null    float64
 7   specialisation  148 non-null    str    
 8   mba_p           148 non-null    float64
 9   salary          148 non-null    float64
dtypes: float64(5), int64(1), str(4)
memory usage: 12.7 KB


,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
0,1,M,0.857051,Commerce,0.057143,Sci&Tech,0.104167,Mkt&HR,58.80,0.094595
1,2,M,0.586729,Science,0.613714,Sci&Tech,0.760417,Mkt&Fin,66.28,0.000000
2,3,M,0.409023,Arts,0.228571,Comm&Mgmt,0.520833,Mkt&Fin,57.80,0.067568
4,5,M,0.485812,Commerce,0.494286,Comm&Mgmt,0.975000,Mkt&Fin,55.50,0.304054
7,8,M,0.280990,Science,0.285714,Sci&Tech,0.354167,Mkt&Fin,62.14,0.070270


## Step 7: Convert categorical variables into numerical format using LabelEncoder ().
[![Picture1.png](https://i.postimg.cc/yNpNvnVd/Picture1.png)](https://postimg.cc/zLW5fCHZ)




In [31]:
for col in ["gender", "hsc_s", "degree_t", "specialisation"]:
    print(col, "->", df[col].unique())


gender -> <StringArray>
['M', 'F']
Length: 2, dtype: str
hsc_s -> <StringArray>
['Commerce', 'Science', 'Arts']
Length: 3, dtype: str
degree_t -> <StringArray>
['Sci&Tech', 'Comm&Mgmt', 'Others']
Length: 3, dtype: str
specialisation -> <StringArray>
['Mkt&HR', 'Mkt&Fin']
Length: 2, dtype: str


## Convert categorical variables into numerical format using one hot encoder

[![Picture2.png](https://i.postimg.cc/gcZ0Hv1J/Picture2.png)](https://postimg.cc/HjTHp7YD)

In [32]:
L1=LabelEncoder()
df["gender"]=L1.fit_transform(df["gender"])
df["hsc_s"]=L1.fit_transform(df["hsc_s"])
df["degree_t"]=L1.fit_transform(df["degree_t"])
df["specialisation"]=L1.fit_transform(df["specialisation"])
df.head()

,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
0,1,1,0.857051,1,0.057143,2,0.104167,1,58.80,0.094595
1,2,1,0.586729,2,0.613714,2,0.760417,0,66.28,0.000000
2,3,1,0.409023,0,0.228571,0,0.520833,0,57.80,0.067568
4,5,1,0.485812,1,0.494286,0,0.975000,0,55.50,0.304054
7,8,1,0.280990,2,0.285714,2,0.354167,0,62.14,0.070270


In [33]:
c=["gender"]
one_hot_encoded_data=pd.get_dummies(df,columns=c)
one_hot_encoded_data.to_csv('clean.csv',index=False)
one_hot_encoded_data.head()

Y=df['salary'].copy()

# Remove target and ID from features
X=df.drop(columns=['salary','sl_no']).copy()

In [34]:
print("X shape:", X.shape)
print("Y shape:", Y.shape)

final_df = X.copy()
final_df["salary"] = Y
print("Missing values in final dataset:\n", final_df.isnull().sum())
print("Final dataset shape:", final_df.shape)

final_df.to_csv("Final.csv", index=False)
final_df.head()


X shape: (148, 8)
Y shape: (148,)
Missing values in final dataset:
 gender            0
hsc_p             0
hsc_s             0
degree_p          0
degree_t          0
etest_p           0
specialisation    0
mba_p             0
salary            0
dtype: int64
Final dataset shape: (148, 9)


,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
0,1,0.857051,1,0.057143,2,0.104167,1,58.80,0.094595
1,1,0.586729,2,0.613714,2,0.760417,0,66.28,0.000000
2,1,0.409023,0,0.228571,0,0.520833,0,57.80,0.067568
4,1,0.485812,1,0.494286,0,0.975000,0,55.50,0.304054
7,1,0.280990,2,0.285714,2,0.354167,0,62.14,0.070270


# Exercise: Data Cleaning and Transformation – Automobile Dataset

## Step 1: Load and Explore the Dataset

### 1. Load the Dataset
- Import Pandas and load the Automobile dataset.
- Display the first 10 rows.
- Display the shape of the dataset.

### 2. Explore the Dataset
- Display the column names.
- Display the data types.
- Generate descriptive statistics.
- Identify numerical and categorical columns.
- Display unique values in categorical columns.

## Step 2: Data Cleaning

### 3. Check Missing Values
- Check for missing values in each column.
- Display the number and percentage of missing values.

### 4. Handle Missing Values
- Replace missing numerical values using mean or median.
- Replace missing categorical values using mode.
- Verify that no missing values remain.

### 5. Remove Duplicate Records
- Check for duplicate rows.
- Display the number of duplicate records.
- Remove duplicate records.
- Verify the result.

### 6. Clean the `horsepower` Column
- Identify non-numeric values such as `?`.
- Replace `?` with `NaN`.
- Convert `horsepower` to numeric.
- Handle the resulting missing values.

## Step 3: Data Transformation

### 7. Transform the `origin` Column
- Display the unique values in `origin`.
- Convert the values into meaningful labels:
  - `1` → `usa`
  - `2` → `europe`
  - `3` → `japan`

### 8. Create `weight_kg`
- Create a new column `weight_kg`.
- Convert weight from pounds to kilograms.

  `weight_kg = weight × 0.453592`

### 9. Create `mpg_category`
Create a new column based on `mpg`:
- `< 20` → `Low`
- `20–29` → `Medium`
- `≥ 30` → `High`

### 10. Create `vehicle_age`
- Create a new column `vehicle_age`.
- Assume the current year is 2026.

  `vehicle_age = 2026 - model_year`

### 11. Rename Columns
Rename:
- `mpg` → `miles_per_gallon`
- `horsepower` → `hp`
- `weight` → `weight_lbs`
- `model_year` → `year`

### 12. Filter the Data
Display vehicles:
- With `mpg > 30`
- With `horsepower > 150`
- With `cylinders >= 6`
- Manufactured after 1980
- Originating from `usa`

## Step 4: Encoding Categorical Data

### 13. Label Encoding
- Apply `LabelEncoder` to the `origin` column.
- Create a new column `origin_encoded`.
- Display the original and encoded values.
- Display the category-to-label mapping.

### 14. One-Hot Encoding
- Apply One-Hot Encoding to the `origin` column.
- Compare Label Encoding and One-Hot Encoding.
- Which encoding method is more appropriate for `origin`? Explain why.

## Step 5: Outlier Detection

### 15. Identify Outliers Using Z-Scores
- Calculate the Z-score for the numerical features.
- Identify observations with `|Z-score| > 3` as outliers.
- Count the outliers in each numerical column.
- Display the rows containing outliers.
- Decide whether the outliers should be removed or retained.

## Step 6: Feature Scaling

### 16. Standardization Using StandardScaler
- Select the numerical features.
- Apply `StandardScaler`.
- Display the standardized values.
- Verify that the features have approximately mean `0` and standard deviation `1`.

## Step 7: Normalization

### 17. Normalization Using MinMaxScaler
- Apply `MinMaxScaler` to the numerical features.
- Transform the features to the range `[0, 1]`.
- Display the normalized values.
- Compare **Standardization** and **Normalization**.
- Explain when each scaling method is appropriate.

## Step 8: Create Features and Target

### 18. Create X and Y Variables
- Select the appropriate input features as **X (independent variables)**.
- Select `mpg` as **Y (target variable)**.
- Display the shape of `X` and `Y`.
- Save `X` and `Y` into `automobile_X_Y.csv`.
- Load the CSV file again and display the first 5 rows.

## Step 9: Save the Final Dataset

### 19. Save the Preprocessed Dataset
- Combine the processed features and target variable.
- Display the final dataset.
- Check for missing values.
- Save the final dataset as `automobile_preprocessed.csv`.

## Step 1: Load and Explore the Dataset

### 1. Load the Dataset

In [35]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from scipy import stats

auto = pd.read_csv("Automobile.csv")
auto.head(10)


,name,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin
0,chevrolet chevelle malibu,18.0,8.0,307.0,130.0,3504.0,12.0,70,usa
1,buick skylark 320,15.0,8.0,350.0,165.0,3693.0,11.5,70,usa
2,plymouth satellite,18.0,8.0,318.0,150.0,3436.0,11.0,70,usa
3,amc rebel sst,16.0,8.0,304.0,150.0,3433.0,12.0,70,usa
4,ford torino,17.0,NaN,302.0,140.0,3449.0,10.5,70,usa
5,ford galaxie 500,15.0,8.0,429.0,198.0,4341.0,10.0,70,usa
6,chevrolet impala,14.0,8.0,454.0,220.0,NaN,9.0,70,usa
7,plymouth fury iii,14.0,8.0,440.0,215.0,4312.0,8.5,70,usa
8,pontiac catalina,14.0,8.0,455.0,NaN,4425.0,10.0,70,usa
9,amc ambassador dpl,15.0,8.0,390.0,190.0,3850.0,8.5,70,usa


In [36]:
auto.shape


(398, 9)

### 2. Explore the Dataset

In [37]:
auto.columns


Index(['name', 'mpg', 'cylinders', 'displacement', 'horsepower', 'weight',
       'acceleration', 'model_year', 'origin'],
      dtype='str')

In [38]:
auto.dtypes


name                str
mpg             float64
cylinders       float64
displacement    float64
horsepower      float64
weight          float64
acceleration    float64
model_year        int64
origin              str
dtype: object

In [39]:
auto.describe(include="all")


,name,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin
count,398,398.000000,395.000000,395.000000,386.000000,396.000000,395.000000,398.000000,398
unique,305,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3
top,ford pinto,NaN,NaN,NaN,NaN,NaN,NaN,NaN,usa
freq,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,249
mean,NaN,23.514573,5.445570,193.340506,104.316062,2965.025253,15.562278,76.010050,NaN
std,NaN,7.815984,1.696203,104.425993,38.086281,845.254458,2.750260,3.697627,NaN
min,NaN,9.000000,3.000000,68.000000,46.000000,1613.000000,8.000000,70.000000,NaN
25%,NaN,17.500000,4.000000,102.500000,75.250000,2222.250000,13.850000,73.000000,NaN
50%,NaN,23.000000,4.000000,146.000000,92.500000,2797.500000,15.500000,76.000000,NaN
75%,NaN,29.000000,8.000000,262.000000,125.000000,3581.750000,17.150000,79.000000,NaN


In [40]:
numerical_cols = auto.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = auto.select_dtypes(include=["object"]).columns.tolist()
print("Numerical columns:", numerical_cols)
print("Categorical columns:", categorical_cols)


Numerical columns: ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year']
Categorical columns: ['name', 'origin']


/tmp/ipykernel_533/1337646303.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = auto.select_dtypes(include=["object"]).columns.tolist()


In [41]:
for col in categorical_cols:
    print(col, "->", auto[col].unique())


name -> <StringArray>
[ 'chevrolet chevelle malibu',          'buick skylark 320',
         'plymouth satellite',              'amc rebel sst',
                'ford torino',           'ford galaxie 500',
           'chevrolet impala',          'plymouth fury iii',
           'pontiac catalina',         'amc ambassador dpl',
 ...
 'chrysler lebaron medallion',             'ford granada l',
           'toyota celica gt',          'dodge charger 2.2',
           'chevrolet camaro',            'ford mustang gl',
                  'vw pickup',              'dodge rampage',
                'ford ranger',                 'chevy s-10']
Length: 305, dtype: str
origin -> <StringArray>
['usa', 'japan', 'europe']
Length: 3, dtype: str


## Step 2: Data Cleaning

### 3. Check Missing Values

In [42]:
missing_count = auto.isnull().sum()
missing_percent = (auto.isnull().sum() / len(auto)) * 100
missing_summary = pd.DataFrame({"Missing Count": missing_count, "Missing %": missing_percent})
missing_summary


,Missing Count,Missing %
name,0,0.000000
mpg,0,0.000000
cylinders,3,0.753769
displacement,3,0.753769
horsepower,12,3.015075
weight,2,0.502513
acceleration,3,0.753769
model_year,0,0.000000
origin,0,0.000000


### 4. Handle Missing Values

In [43]:
# Replace missing numerical values using the median
num_missing_cols = ["cylinders", "displacement", "horsepower", "weight", "acceleration"]
for col in num_missing_cols:
    if col in auto.columns:
        auto[col] = auto[col].fillna(auto[col].median())


In [44]:
# Replace missing categorical values using the mode
for col in categorical_cols:
    if auto[col].isnull().sum() > 0:
        auto[col] = auto[col].fillna(auto[col].mode()[0])


In [45]:
# Verify no missing values remain
auto.isnull().sum()


name            0
mpg             0
cylinders       0
displacement    0
horsepower      0
weight          0
acceleration    0
model_year      0
origin          0
dtype: int64

### 5. Remove Duplicate Records

In [46]:
duplicate_count = auto.duplicated().sum()
print("Number of duplicate rows:", duplicate_count)


Number of duplicate rows: 0


In [47]:
auto.drop_duplicates(inplace=True)
print("Duplicates after removal:", auto.duplicated().sum())
auto.shape


Duplicates after removal: 0


(398, 9)

### 6. Clean the `horsepower` Column

In [48]:
# Identify non-numeric values such as "?"
print(auto["horsepower"].unique())


[130.  165.  150.  140.  198.  220.  215.   92.5 190.  170.  160.  225.
  95.   97.   85.   88.   87.   90.  113.  200.  210.  193.  100.  105.
 175.  153.  180.  110.   72.   86.   70.   76.   65.   69.   60.   80.
  54.  208.  155.  112.   92.  145.  137.  158.   46.  167.   94.  107.
 230.   49.   75.   91.  122.   67.   83.   78.   52.   61.   93.  148.
 129.   96.   71.   98.  115.   53.   81.   79.  120.  152.  102.  108.
  68.   58.  149.   89.   63.   48.   66.  139.  103.  125.  133.  138.
 135.  142.   77.   62.  132.   84.   64.   74.  116.   82. ]


In [49]:
# Replace "?" with NaN and convert to numeric
auto["horsepower"] = auto["horsepower"].replace("?", np.nan)
auto["horsepower"] = pd.to_numeric(auto["horsepower"], errors="coerce")

# Handle the resulting missing values using the median
auto["horsepower"] = auto["horsepower"].fillna(auto["horsepower"].median())
auto["horsepower"].isnull().sum()


np.int64(0)

## Step 3: Data Transformation

### 7. Transform the `origin` Column

In [50]:
print(auto["origin"].unique())


<StringArray>
['usa', 'japan', 'europe']
Length: 3, dtype: str


In [51]:
# Map numeric codes to meaningful labels if origin is still numeric
origin_map = {1: "usa", 2: "europe", 3: "japan"}
if pd.api.types.is_numeric_dtype(auto["origin"]):
    auto["origin"] = auto["origin"].map(origin_map)
auto["origin"].unique()


<StringArray>
['usa', 'japan', 'europe']
Length: 3, dtype: str

### 8. Create `weight_kg`

In [52]:
auto["weight_kg"] = auto["weight"] * 0.453592
auto[["weight", "weight_kg"]].head()


,weight,weight_kg
0,3504.0,1589.386368
1,3693.0,1675.115256
2,3436.0,1558.542112
3,3433.0,1557.181336
4,3449.0,1564.438808


### 9. Create `mpg_category`

In [53]:
def mpg_category(mpg):
    if mpg < 20:
        return "Low"
    elif mpg < 30:
        return "Medium"
    else:
        return "High"

auto["mpg_category"] = auto["mpg"].apply(mpg_category)
auto[["mpg", "mpg_category"]].head()


,mpg,mpg_category
0,18.0,Low
1,15.0,Low
2,18.0,Low
3,16.0,Low
4,17.0,Low


### 10. Create `vehicle_age`

In [54]:
current_year = 2026
auto["vehicle_age"] = current_year - (1900 + auto["model_year"])
auto[["model_year", "vehicle_age"]].head()


,model_year,vehicle_age
0,70,56
1,70,56
2,70,56
3,70,56
4,70,56


### 11. Rename Columns

In [55]:
auto.rename(columns={
    "mpg": "miles_per_gallon",
    "horsepower": "hp",
    "weight": "weight_lbs",
    "model_year": "year"
}, inplace=True)
auto.columns


Index(['name', 'miles_per_gallon', 'cylinders', 'displacement', 'hp',
       'weight_lbs', 'acceleration', 'year', 'origin', 'weight_kg',
       'mpg_category', 'vehicle_age'],
      dtype='str')

### 12. Filter the Data

In [56]:
# Vehicles with miles_per_gallon > 30
auto[auto["miles_per_gallon"] > 30].head()


,name,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration,year,origin,weight_kg,mpg_category,vehicle_age
53,toyota corolla 1200,31.0,4.0,71.0,65.0,1773.0,19.0,71,japan,804.218616,High,55
54,datsun 1200,35.0,4.0,72.0,69.0,1613.0,18.0,71,japan,731.643896,High,55
129,datsun b210,31.0,4.0,79.0,67.0,1950.0,19.0,74,japan,884.504400,High,52
131,toyota corolla 1200,32.0,4.0,71.0,92.5,1836.0,15.5,74,japan,832.794912,High,52
144,toyota corona,31.0,4.0,76.0,52.0,1649.0,16.5,74,japan,747.973208,High,52


In [57]:
# Vehicles with hp > 150
auto[auto["hp"] > 150].head()


,name,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration,year,origin,weight_kg,mpg_category,vehicle_age
1,buick skylark 320,15.0,8.0,350.0,165.0,3693.0,11.5,70,usa,1675.115256,Low,56
5,ford galaxie 500,15.0,8.0,429.0,198.0,4341.0,10.0,70,usa,1969.042872,Low,56
6,chevrolet impala,14.0,8.0,454.0,220.0,2797.5,9.0,70,usa,1268.923620,Low,56
7,plymouth fury iii,14.0,8.0,440.0,215.0,4312.0,8.5,70,usa,1955.888704,Low,56
9,amc ambassador dpl,15.0,8.0,390.0,190.0,3850.0,8.5,70,usa,1746.329200,Low,56


In [58]:
# Vehicles with cylinders >= 6
auto[auto["cylinders"] >= 6].head()


,name,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration,year,origin,weight_kg,mpg_category,vehicle_age
0,chevrolet chevelle malibu,18.0,8.0,307.0,130.0,3504.0,12.0,70,usa,1589.386368,Low,56
1,buick skylark 320,15.0,8.0,350.0,165.0,3693.0,11.5,70,usa,1675.115256,Low,56
2,plymouth satellite,18.0,8.0,318.0,150.0,3436.0,11.0,70,usa,1558.542112,Low,56
3,amc rebel sst,16.0,8.0,304.0,150.0,3433.0,12.0,70,usa,1557.181336,Low,56
5,ford galaxie 500,15.0,8.0,429.0,198.0,4341.0,10.0,70,usa,1969.042872,Low,56


In [59]:
# Vehicles manufactured after 1980 (year stored as 2-digit, e.g. 80 = 1980)
auto[auto["year"] > 80].head()


,name,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration,year,origin,weight_kg,mpg_category,vehicle_age
338,plymouth reliant,27.2,4.0,135.0,84.0,2490.0,15.7,81,usa,1129.44408,Medium,45
339,buick skylark,26.6,4.0,151.0,84.0,2635.0,16.4,81,usa,1195.21492,Medium,45
340,dodge aries wagon (sw),25.8,4.0,156.0,92.0,2620.0,14.4,81,usa,1188.41104,Medium,45
341,chevrolet citation,23.5,6.0,173.0,110.0,2725.0,12.6,81,usa,1236.03820,Medium,45
342,plymouth reliant,30.0,4.0,135.0,84.0,2385.0,12.9,81,usa,1081.81692,High,45


In [60]:
# Vehicles originating from usa
auto[auto["origin"] == "usa"].head()


,name,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration,year,origin,weight_kg,mpg_category,vehicle_age
0,chevrolet chevelle malibu,18.0,8.0,307.0,130.0,3504.0,12.0,70,usa,1589.386368,Low,56
1,buick skylark 320,15.0,8.0,350.0,165.0,3693.0,11.5,70,usa,1675.115256,Low,56
2,plymouth satellite,18.0,8.0,318.0,150.0,3436.0,11.0,70,usa,1558.542112,Low,56
3,amc rebel sst,16.0,8.0,304.0,150.0,3433.0,12.0,70,usa,1557.181336,Low,56
4,ford torino,17.0,4.0,302.0,140.0,3449.0,10.5,70,usa,1564.438808,Low,56


## Step 4: Encoding Categorical Data

### 13. Label Encoding

In [61]:
le = LabelEncoder()
auto["origin_encoded"] = le.fit_transform(auto["origin"])
auto[["origin", "origin_encoded"]].drop_duplicates()


,origin,origin_encoded
0,usa,2
14,japan,1
19,europe,0


In [62]:
mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("Category to label mapping:", mapping)


Category to label mapping: {'europe': np.int64(0), 'japan': np.int64(1), 'usa': np.int64(2)}


### 14. One-Hot Encoding

In [63]:
auto_one_hot = pd.get_dummies(auto, columns=["origin"], prefix="origin")
auto_one_hot.head()


,name,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration,year,weight_kg,mpg_category,vehicle_age,origin_encoded,origin_europe,origin_japan,origin_usa
0,chevrolet chevelle malibu,18.0,8.0,307.0,130.0,3504.0,12.0,70,1589.386368,Low,56,2,False,False,True
1,buick skylark 320,15.0,8.0,350.0,165.0,3693.0,11.5,70,1675.115256,Low,56,2,False,False,True
2,plymouth satellite,18.0,8.0,318.0,150.0,3436.0,11.0,70,1558.542112,Low,56,2,False,False,True
3,amc rebel sst,16.0,8.0,304.0,150.0,3433.0,12.0,70,1557.181336,Low,56,2,False,False,True
4,ford torino,17.0,4.0,302.0,140.0,3449.0,10.5,70,1564.438808,Low,56,2,False,False,True


**Label Encoding vs One-Hot Encoding:** Label Encoding assigns each category an arbitrary integer (e.g. usa=2, europe=0, japan=1). This introduces a false sense of order/magnitude between categories that don't have a natural rank. One-Hot Encoding creates a separate binary column for each category, so no ordinal relationship is implied.

**Which is more appropriate for `origin`?** One-Hot Encoding is more appropriate, because `origin` is a nominal (unordered) categorical variable — there is no inherent ranking between `usa`, `europe`, and `japan`. Using Label Encoding could mislead a machine learning model into assuming an ordinal relationship (e.g. that japan > europe > usa) that doesn't actually exist.


## Step 5: Outlier Detection

### 15. Identify Outliers Using Z-Scores

In [64]:
numeric_features = ["miles_per_gallon", "cylinders", "displacement", "hp",
                    "weight_lbs", "acceleration"]
z_scores = auto[numeric_features].apply(stats.zscore)
z_scores.head()


,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration
0,-0.706439,1.515897,1.096516,0.694154,0.641001,-1.301636
1,-1.090751,1.515897,1.510055,1.627150,0.865428,-1.484357
2,-0.706439,1.515897,1.202305,1.227295,0.560255,-1.667078
3,-0.962647,1.515897,1.067664,1.227295,0.556693,-1.301636
4,-0.834543,-0.847774,1.048430,0.960725,0.575692,-1.849799


In [65]:
outlier_mask = (z_scores.abs() > 3)
outlier_counts = outlier_mask.sum()
print("Outlier count per column:\n", outlier_counts)


Outlier count per column:
 miles_per_gallon    0
cylinders           0
displacement        0
hp                  4
weight_lbs          0
acceleration        2
dtype: int64


In [66]:
rows_with_outliers = auto[outlier_mask.any(axis=1)]
rows_with_outliers


,name,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration,year,origin,weight_kg,mpg_category,vehicle_age,origin_encoded
6,chevrolet impala,14.0,8.0,454.0,220.0,2797.5,9.0,70,usa,1268.923620,Low,56,2
13,buick estate wagon (sw),14.0,8.0,455.0,225.0,3086.0,10.0,70,usa,1399.784912,Low,56,2
95,buick electra 225 custom,12.0,8.0,455.0,225.0,4951.0,11.0,73,usa,2245.733992,Low,53,2
116,pontiac grand prix,16.0,8.0,400.0,230.0,4278.0,9.5,73,usa,1940.466576,Low,53,2
299,peugeot 504,27.2,4.0,141.0,71.0,3190.0,24.8,79,europe,1446.958480,Medium,47,0
394,vw pickup,44.0,4.0,97.0,52.0,2130.0,24.6,82,europe,966.150960,High,44,0


**Remove or retain?** Only a handful of rows (mostly in `hp` and `acceleration`) exceed the |Z| > 3 threshold. Since these values are genuine (not data-entry errors) and the dataset is already small, it's safer to **retain** them — removing them would throw away valid high-performance / low-performance vehicles. If a model turns out to be very sensitive to these extreme values, they can be capped instead of dropped.


## Step 6: Feature Scaling

### 16. Standardization Using StandardScaler

In [67]:
auto_scaled = auto.copy()
scaler = StandardScaler()
auto_scaled[numeric_features] = scaler.fit_transform(auto_scaled[numeric_features])
auto_scaled[numeric_features].head()


,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration
0,-0.706439,1.515897,1.096516,0.694154,0.641001,-1.301636
1,-1.090751,1.515897,1.510055,1.627150,0.865428,-1.484357
2,-0.706439,1.515897,1.202305,1.227295,0.560255,-1.667078
3,-0.962647,1.515897,1.067664,1.227295,0.556693,-1.301636
4,-0.834543,-0.847774,1.048430,0.960725,0.575692,-1.849799


In [68]:
print("Mean after scaling:\n", auto_scaled[numeric_features].mean().round(2))
print("\nStd after scaling:\n", auto_scaled[numeric_features].std().round(2))


Mean after scaling:
 miles_per_gallon    0.0
cylinders          -0.0
displacement       -0.0
hp                  0.0
weight_lbs         -0.0
acceleration       -0.0
dtype: float64

Std after scaling:
 miles_per_gallon    1.0
cylinders           1.0
displacement        1.0
hp                  1.0
weight_lbs          1.0
acceleration        1.0
dtype: float64


## Step 7: Normalization

### 17. Normalization Using MinMaxScaler

In [69]:
auto_normalized = auto.copy()
minmax = MinMaxScaler()
auto_normalized[numeric_features] = minmax.fit_transform(auto_normalized[numeric_features])
auto_normalized[numeric_features].head()


,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration
0,0.239362,1.0,0.617571,0.456522,0.536150,0.238095
1,0.159574,1.0,0.728682,0.646739,0.589736,0.208333
2,0.239362,1.0,0.645995,0.565217,0.516870,0.178571
3,0.186170,1.0,0.609819,0.565217,0.516019,0.238095
4,0.212766,0.2,0.604651,0.510870,0.520556,0.148810


**Standardization vs Normalization:** Standardization (StandardScaler) rescales data to have a mean of 0 and standard deviation of 1, and works well when the data roughly follows a normal distribution or when the model (e.g. linear regression, logistic regression, SVM, PCA) assumes/benefits from standardized input. Normalization (MinMaxScaler) rescales data to a fixed range `[0, 1]`, and is preferred when the data doesn't follow a normal distribution, when the algorithm needs bounded inputs (e.g. neural networks, image pixel data, distance-based algorithms like KNN), or when preserving the original relative spacing between the smallest and largest values matters.


## Step 8: Create Features and Target

### 18. Create X and Y Variables

In [70]:
Y_auto = auto["miles_per_gallon"].copy()
X_auto = auto.drop(columns=["miles_per_gallon", "name", "mpg_category"]).copy()

# One-hot encode the remaining categorical column (origin) for a model-ready X
X_auto = pd.get_dummies(X_auto, columns=["origin"], prefix="origin")

print("X shape:", X_auto.shape)
print("Y shape:", Y_auto.shape)


X shape: (398, 12)
Y shape: (398,)


In [71]:
auto_X_Y = X_auto.copy()
auto_X_Y["mpg"] = Y_auto
auto_X_Y.to_csv("automobile_X_Y.csv", index=False)

reloaded = pd.read_csv("automobile_X_Y.csv")
reloaded.head()


,cylinders,displacement,hp,weight_lbs,acceleration,year,weight_kg,vehicle_age,origin_encoded,origin_europe,origin_japan,origin_usa,mpg
0,8.0,307.0,130.0,3504.0,12.0,70,1589.386368,56,2,False,False,True,18.0
1,8.0,350.0,165.0,3693.0,11.5,70,1675.115256,56,2,False,False,True,15.0
2,8.0,318.0,150.0,3436.0,11.0,70,1558.542112,56,2,False,False,True,18.0
3,8.0,304.0,150.0,3433.0,12.0,70,1557.181336,56,2,False,False,True,16.0
4,4.0,302.0,140.0,3449.0,10.5,70,1564.438808,56,2,False,False,True,17.0


## Step 9: Save the Final Dataset

### 19. Save the Preprocessed Dataset

In [72]:
automobile_final = X_auto.copy()
automobile_final["mpg"] = Y_auto
automobile_final.head()


,cylinders,displacement,hp,weight_lbs,acceleration,year,weight_kg,vehicle_age,origin_encoded,origin_europe,origin_japan,origin_usa,mpg
0,8.0,307.0,130.0,3504.0,12.0,70,1589.386368,56,2,False,False,True,18.0
1,8.0,350.0,165.0,3693.0,11.5,70,1675.115256,56,2,False,False,True,15.0
2,8.0,318.0,150.0,3436.0,11.0,70,1558.542112,56,2,False,False,True,18.0
3,8.0,304.0,150.0,3433.0,12.0,70,1557.181336,56,2,False,False,True,16.0
4,4.0,302.0,140.0,3449.0,10.5,70,1564.438808,56,2,False,False,True,17.0


In [73]:
print("Missing values in final dataset:\n", automobile_final.isnull().sum())


Missing values in final dataset:
 cylinders         0
displacement      0
hp                0
weight_lbs        0
acceleration      0
year              0
weight_kg         0
vehicle_age       0
origin_encoded    0
origin_europe     0
origin_japan      0
origin_usa        0
mpg               0
dtype: int64


In [74]:
automobile_final.to_csv("automobile_preprocessed.csv", index=False)
print("Saved automobile_preprocessed.csv with shape:", automobile_final.shape)


Saved automobile_preprocessed.csv with shape: (398, 13)
